In [0]:
%pip install pypdf databricks-vectorsearch sentence-transformers --quiet
dbutils.library.restartPython()

In [0]:
from pypdf import PdfReader
import os

#Cell 2: 49 general chunks, 9 restricted chunks extracted — reasonable for 8 short docs + 2 short docs#
CATALOG = "employee_management"
SCHEMA = "fullstack"

def extract_and_chunk(volume_path, chunk_size=800, overlap=100):
    records = []
    for fname in os.listdir(volume_path):
        if not fname.lower().endswith(".pdf"):
            continue
        full_path = os.path.join(volume_path, fname)
        reader = PdfReader(full_path)
        full_text = ""
        for page in reader.pages:
            full_text += page.extract_text() + "\n"

        start = 0
        chunk_id = 0
        while start < len(full_text):
            chunk = full_text[start:start + chunk_size]
            if chunk.strip():
                records.append({
                    "chunk_id": f"{fname}_{chunk_id}",
                    "source_file": fname,
                    "content": chunk.strip()
                })
            start += chunk_size - overlap
            chunk_id += 1
    return records

general_records = extract_and_chunk(f"/Volumes/{CATALOG}/{SCHEMA}/general_docs")
restricted_records = extract_and_chunk(f"/Volumes/{CATALOG}/{SCHEMA}/restricted_docs")

print(f"General chunks: {len(general_records)}")
print(f"Restricted chunks: {len(restricted_records)}")

In [0]:
from pyspark.sql import Row
#Both Delta tables created, CDC enabled — "Tables created and CDC enabled" printed
general_df = spark.createDataFrame([Row(**r) for r in general_records])
restricted_df = spark.createDataFrame([Row(**r) for r in restricted_records])

general_df.write.format("delta").mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.general_docs_chunks"
)
restricted_df.write.format("delta").mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.restricted_docs_chunks"
)

spark.sql(f"ALTER TABLE {CATALOG}.{SCHEMA}.general_docs_chunks SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
spark.sql(f"ALTER TABLE {CATALOG}.{SCHEMA}.restricted_docs_chunks SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

print("Tables created and CDC enabled.")

In [0]:
from databricks.vector_search.client import VectorSearchClient
#Cell 4: Endpoint employee_ai_endpoint created successfully
#✅ Compute → AI Search screenshot: status shows Ready (green), 0 indexes — that's expected, you haven't created the indexes yet, that's what Cell 5 does#
vsc = VectorSearchClient()
ENDPOINT_NAME = "employee_ai_endpoint"

existing = [e["name"] for e in vsc.list_endpoints().get("endpoints", [])]
if ENDPOINT_NAME not in existing:
    vsc.create_endpoint(name=ENDPOINT_NAME, endpoint_type="STANDARD")
    print("Creating endpoint... this can take a few minutes.")
else:
    print("Endpoint already exists.")

In [0]:
#deleting broken indexes
#vsc.delete_index(ENDPOINT_NAME, f"{CATALOG}.{SCHEMA}.general_docs_index")
#vsc.delete_index(ENDPOINT_NAME, f"{CATALOG}.{SCHEMA}.restricted_docs_index")
#print("Old indexes deleted.")

In [0]:
#compute embedding locally and creating direct access inndex

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")  # 384-dim, runs on CPU, no external API

def embed_records(records):
    texts = [r["content"] for r in records]
    embeddings = model.encode(texts, show_progress_bar=True).tolist()
    for r, emb in zip(records, embeddings):
        r["embedding"] = emb
    return records

general_records = embed_records(general_records)
restricted_records = embed_records(restricted_records)

print("Embeddings computed for", len(general_records), "general and", len(restricted_records), "restricted chunks.")


In [0]:
#create direct access index and uploading vectors
import time

# Delete existing indexes if they exist
existing_indexes = [idx["name"] for idx in vsc.list_indexes(ENDPOINT_NAME).get("vector_indexes", [])]
if f"{CATALOG}.{SCHEMA}.general_docs_index" in existing_indexes:
    vsc.delete_index(endpoint_name=ENDPOINT_NAME, index_name=f"{CATALOG}.{SCHEMA}.general_docs_index")
    print("Deleting general_docs_index...")
if f"{CATALOG}.{SCHEMA}.restricted_docs_index" in existing_indexes:
    vsc.delete_index(endpoint_name=ENDPOINT_NAME, index_name=f"{CATALOG}.{SCHEMA}.restricted_docs_index")
    print("Deleting restricted_docs_index...")

# Wait for deletions to complete
if existing_indexes:
    print("Waiting for deletions to complete...")
    time.sleep(10)
    # Verify deletions completed
    for _ in range(12):  # Wait up to 2 minutes
        current_indexes = [idx["name"] for idx in vsc.list_indexes(ENDPOINT_NAME).get("vector_indexes", [])]
        if (f"{CATALOG}.{SCHEMA}.general_docs_index" not in current_indexes and 
            f"{CATALOG}.{SCHEMA}.restricted_docs_index" not in current_indexes):
            break
        time.sleep(10)
    print("Deletions complete.")

general_index = vsc.create_direct_access_index(
    endpoint_name=ENDPOINT_NAME,
    index_name=f"{CATALOG}.{SCHEMA}.general_docs_index",
    primary_key="chunk_id",
    embedding_dimension=384,
    embedding_vector_column="embedding",
    schema={"chunk_id": "string", "source_file": "string", "content": "string", "embedding": "array<float>"}
)
print("Waiting for general_docs_index to be ready...")
for _ in range(30):  # Wait up to 5 minutes
    try:
        idx_info = vsc.get_index(endpoint_name=ENDPOINT_NAME, index_name=f"{CATALOG}.{SCHEMA}.general_docs_index")
        if idx_info.describe().get("status", {}).get("state") == "ONLINE":
            break
    except:
        pass
    time.sleep(10)
general_index.upsert(general_records)
print("general_docs_index populated.")

restricted_index = vsc.create_direct_access_index(
    endpoint_name=ENDPOINT_NAME,
    index_name=f"{CATALOG}.{SCHEMA}.restricted_docs_index",
    primary_key="chunk_id",
    embedding_dimension=384,
    embedding_vector_column="embedding",
    schema={"chunk_id": "string", "source_file": "string", "content": "string", "embedding": "array<float>"}
)
print("Waiting for restricted_docs_index to be ready...")
for _ in range(30):  # Wait up to 5 minutes
    try:
        idx_info = vsc.get_index(endpoint_name=ENDPOINT_NAME, index_name=f"{CATALOG}.{SCHEMA}.restricted_docs_index")
        if idx_info.describe().get("status", {}).get("state") == "ONLINE":
            break
    except:
        pass
    time.sleep(10)
restricted_index.upsert(restricted_records)
print("restricted_docs_index populated.")

print("Both indexes created and populated directly — no external embedding API used.")

In [0]:
print("Unique files in restricted_records:", set(r["source_file"] for r in restricted_records))

In [0]:
query_vector = model.encode(["What are the promotion rules?"]).tolist()[0]
results = restricted_index.similarity_search(
    query_vector=query_vector,
    columns=["source_file", "content"],
    num_results=3
)
for r in results["result"]["data_array"]:
    print(r[0], "→", r[1][:150])